# Seção 12 (aplicações) — pegadinhas, efeitos colaterais e caveats

Complemento da seção: o quadro abaixo reúne as **notas de revisão** já registradas nos notebooks desta pasta, e em seguida vêm
pegadinhas **novas**, cada uma com demonstração executável e a forma de evitar. Teoria da seção: [`00_teoria.md`](00_teoria.md).

## Já registradas nesta seção

| notebook | registro |
|---|---|
| [`02_consumindo_uma_api_rest`](02_consumindo_uma_api_rest.ipynb) | validador *after* chama `.casefold()` em `None`; `raise_for_status` sem parênteses; sem `timeout=` |
| [`03_ingerindo_um_csv`](03_ingerindo_um_csv.ipynb) | mensagem de erro sem o "not"; abrir com `newline=""`; erro sem número de linha |
| [`04_validando_argumentos_de_funcoes`](04_validando_argumentos_de_funcoes.ipynb) | versão "Python puro" anota `datetime \| str` e quebra com `datetime` |
| [`05_geradores_de_codigo_de_modelos`](05_geradores_de_codigo_de_modelos.ipynb) | `pattern` sem âncora; o schema 3 não exige o campo que o modelo exige; `--reuse-model`; células de código com prosa |

## Novas

**1. Resposta de erro em HTML: `response.json()` explode com `JSONDecodeError`.** Proxies e balanceadores respondem 502/503 com uma
página HTML. Simulando a resposta (sem rede):

In [1]:
import csv
import io
import shutil
import subprocess
import tempfile
from pathlib import Path

import requests
from pydantic import validate_call

gateway_error = requests.Response()
gateway_error.status_code = 502
gateway_error._content = b"<html><body>Bad Gateway</body></html>"
try:
    gateway_error.json()
except requests.exceptions.JSONDecodeError as ex:
    print(f"response.json(): JSONDecodeError: {ex}")
try:
    gateway_error.raise_for_status()
except requests.HTTPError as ex:
    print(f"raise_for_status(): HTTPError: {ex}")

response.json(): JSONDecodeError: Expecting value: line 1 column 1 (char 0)
raise_for_status(): HTTPError: 502 Server Error: None for url: None


Sem o `raise_for_status()` **antes** do `.json()`, o log mostra "erro de JSON" em vez de "502 do servidor", e a investigação vai
para o lugar errado. Ordem: status → conteúdo (`Content-Type`) → parse → validação.

**2. CSV com BOM: a primeira coluna ganha um caractere invisível.** Planilhas exportadas pelo Excel costumam começar com `﻿`:

In [2]:
with tempfile.TemporaryDirectory() as tmp:
    exported = Path(tmp, "export.csv")
    exported.write_bytes("﻿area;populacao\nAlabama;4464356\n".encode("utf-8"))
    for encoding in ["utf-8", "utf-8-sig"]:
        with open(exported, newline="", encoding=encoding) as f:
            header = list(next(csv.DictReader(f, delimiter=";")))
        print(f"{encoding}: chaves {', '.join(repr(key) for key in header)}")

utf-8: chaves '\ufeffarea', 'populacao'
utf-8-sig: chaves 'area', 'populacao'


Com `utf-8`, a chave vira `'﻿area'` e o modelo acusa "area: Field required" com a coluna **visivelmente** presente.
Para arquivos que podem vir do Excel: `encoding="utf-8-sig"` (lê com ou sem BOM).

**3. `@validate_call` não valida defaults.**

In [3]:
@validate_call
def fetch_page(page_size: int = "dez") -> int:
    return page_size


print(f"fetch_page(): {fetch_page()!r}")
print(f"fetch_page(page_size='10'): {fetch_page(page_size='10')!r}")

fetch_page(): 'dez'
fetch_page(page_size='10'): 10


O default errado só aparece quando alguém usa o valor (`page_size * 2` vira `"dezdez"`). `@validate_call(config={"validate_default": True})`
— ou testes que chamam a função sem argumentos.

**4. `datamodel-codegen` numera nomes que colidem — pela ordem do arquivo.**

In [4]:
def codegen_json(sample: str) -> str:
    exe = shutil.which("datamodel-codegen")
    if exe is None:
        return "# datamodel-codegen não instalado"
    with tempfile.TemporaryDirectory() as tmp:
        source = Path(tmp, "sample.json")
        source.write_text(sample)
        result = subprocess.run([exe, "--input", str(source), "--input-file-type", "json", "--output-model-type",
                                 "pydantic_v2.BaseModel", "--snake-case-field", "--disable-timestamp"],
                                capture_output=True, text=True, check=True)
    return "\n".join(line for line in result.stdout.splitlines() if line.startswith("    ") and ": int" in line)


camel_first = '{"userId": 1, "user_id": 2}'
snake_first = '{"user_id": 2, "userId": 1}'
print(f"ordem userId, user_id:\n{codegen_json(camel_first)}")
print(f"ordem user_id, userId:\n{codegen_json(snake_first)}")

ordem userId, user_id:
    user_id: int = Field(..., alias='userId')
    user_id_1: int = Field(..., alias='user_id')


ordem user_id, userId:
    user_id: int
    user_id_1: int = Field(..., alias='userId')


Os dois campos viram `user_id`, então o gerador numera — e **qual** vira `user_id_1` depende da ordem das chaves no arquivo de
exemplo. Regerar a partir de outro exemplo troca os nomes e quebra o código que os usa. Revise colisões e fixe os nomes com aliases
explícitos depois de gerar.

## Veja também

- [`pydantic/03_pydantic_avancado`](../../03_pydantic_avancado.ipynb) (caveats): `@validate_call` valida uma cópia (mutações não chegam
  ao chamador).
- [`dataclass_factory/01_formatos_de_configuracao`](../../../dataclass_factory/01_formatos_de_configuracao.ipynb) e
  [`06_caso_completo`](../../../dataclass_factory/06_caso_completo.ipynb) (caveats): CSV irregular, números em formato brasileiro, efeitos parciais.